<div dir="rtl">

# HM_05 - מעבדה: לבדוק מה כלי AI החזיר, ולבחור את הכלי הנכון

**הקורס:** מודלי שפה וסוכנים ממוחשבים למסלול מערכות מידע (4261310) ולמסלול ידע ונתונים (תשפז_ב_4261110) · מאת ד״ר ולד לנדא וד״ר רן דובין · **מפגש 5:** בחירת הכלי הנכון למשימה

**שם:** _(כתבו כאן)_

---

## מה עושים במעבדה

במפגש 5 למדנו שלוש משפחות של כלים (**צ'אט, עוזר בעורך, סוכן שורת פקודה**), שכלי הוא **מודל + מעטפת**, ושבוחרים כלי לפי **חמישה קריטריונים** ולא לפי הרעש. במעבדה הזו לא תשוו כלים "בהרגשה": תכתבו בעצמכם את הכלים שמודדים אותם - **מבחן שבודק תוצאה של AI, מנתח של תמלול סוכן, וכלל החלטה לבחירת כלי**.

כל העבודה מתבצעת ב-Colab, על הנתונים הסינתטיים של "החנות". המשימה שמלווה אותנו (משימת העוגן): **למצוא את 5 המוצרים עם ההכנסה הגבוהה ביותר**, כאשר הכנסה = כמות × מחיר.

| חלק | נושא | מה מממשים | זמן |
|---|---|---|---|
| א' | לבדוק לפני שסומכים | `add_revenue`, `top_n_products`, `validate_top5` | 13 דק' |
| ב' | סוכן: מה הוא קרא, יצר והריץ | `summarize_transcript`, `count_approvals`, `find_risky` | 11 דק' |
| ג' | בחירת כלי לפי חמישה קריטריונים | `choose_tool`, סיווג פרטיות | 10 דק' |
| ד' | יומן שימוש ב-AI | `log_row`, `write_log` | 3 דק' |
| ה' | מסקנה | - | 3 דק' |
| העשרה | כלי SQL בסגנון MCP (אופציונלי, בית) | `is_read_only`, `tables_not_allowed`, `safe_query` | ~10 דק' |

## איך עובדים

1. **מממשים** כל פונקציה שמסומנת `# TODO` - **בעצמכם, בלי כלי AI**. המטרה היא להבין מה בודקים, לא לקבל קוד. (רק בצעד 5 בחלק א' משתמשים בכלי AI, כי הוא נושא הניסוי.)
2. **מריצים את תא הבדיקה** שאחרי כל פונקציה. כשהמימוש נכון יודפס `✓`; אחרת תופיע הודעה שמסבירה מה נכשל.
3. **לפני כל מדידה כותבים תחזית** בתא המיועד לכך. תחזית "שגויה" עם ניתוח טוב שווה יותר מתחזית "נכונה" בלי הסבר.
4. **עונים על שאלות הניתוח** בתאי ה"תשובה", במונחים של מה שלמדנו בהרצאה.

> **אל תיבהלו מהקוד.** כל פונקציה כאן היא 2-6 שורות, ובכל שלב מופיעים הסבר, דוגמה ורמז. אם נתקעתם: קראו שוב את ה-docstring (הטקסט בין שלוש המרכאות), ובקשו עזרה מבן הזוג.

## הגשה

בסוף: **Runtime ← Restart session and run all**, ודאו שתא הבדיקה המסכם בסוף המחברת מראה `✓` לכל הפונקציות, ואז **File ← Download ← Download .ipynb** והגישו את הקובץ.

</div>

In [ ]:
# תא הכנה - הריצו פעם אחת בתחילת העבודה
import pandas as pd
import matplotlib.pyplot as plt

print("✓ מוכן")

In [ ]:
#@title תא הכנה - הנתונים (אפשר לא לפתוח) { display-mode: "form" }
import base64, gzip, pathlib
_FILES = {
    'orders.csv': (
        '...'
    ),
}
for _name, _b64 in _FILES.items():
    pathlib.Path(_name).parent.mkdir(parents=True, exist_ok=True)
    pathlib.Path(_name).write_bytes(gzip.decompress(base64.b64decode(_b64)))
print('✓ הקבצים נוצרו:', ', '.join(_FILES))

<div dir="rtl">

---

# חלק א' - לבדוק לפני שסומכים (13 דק')

## הרקע

בהרצאה ביקשנו את אותה משימה משלושה כלים. כל אחד החזיר תוצאה שנראית סבירה. **איך יודעים מי צודק?** לא בהרגשה, ולא ב"זה רץ בלי שגיאה". כותבים **מבחן**: דברים שאנחנו כבר יודעים שחייבים להיות נכונים בתשובה, ובודקים אותם בקוד.

## הנתונים

הקובץ `orders.csv` הוא 3,000 הזמנות סינתטיות מ-2025. כל שורה היא הזמנה של מוצר אחד: `order_id, order_date, customer_id, product_id, product_name, quantity, unit_price`. הריצו וראו:

</div>

In [ ]:
orders = pd.read_csv("orders.csv")
print(len(orders), "שורות")
orders.head()

<div dir="rtl">

## שלב 1 - עמודת ההכנסה

**הכנסה = כמות × מחיר ליחידה** (`quantity * unit_price`). ממשו פונקציה שמקבלת טבלה ומחזירה **עותק** שלה עם עמודה חדשה בשם `revenue`.

**כלים:** `df.copy()` יוצר עותק (כדי לא לשנות את הטבלה המקורית). `out["revenue"] = out["quantity"] * out["unit_price"]` יוצר עמודה חדשה - הכפל מתבצע שורה-שורה.

</div>

In [ ]:
def add_revenue(df):
    """Returns a COPY of df with a new column 'revenue' = quantity * unit_price.
    e.g. a row with quantity=2 and unit_price=10.0 gets revenue=20.0
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - שלב 1
def check_a1():
    t = pd.DataFrame({"quantity": [2, 1], "unit_price": [10.0, 5.0]})
    r = add_revenue(t)
    assert list(r["revenue"]) == [20.0, 5.0], "add_revenue: revenue צריך להיות quantity * unit_price"
    assert "revenue" not in t.columns, "add_revenue: אסור לשנות את הטבלה המקורית - החזירו עותק (copy)"
    return True

check_a1() and print("✓ שלב 1: עמודת revenue נוספה")

<div dir="rtl">

## שלב 2 - חמשת המוצרים המובילים

ממשו פונקציה שמקבלת טבלה **שכבר יש בה עמודת `revenue`** ומחזירה `Series` (עמודה עם שמות): **סך ההכנסה של כל מוצר** (לפי `product_name`), ממוין **מהגבוה לנמוך**, ורק ה-`n` הראשונים.

**כלים:** `df.groupby("product_name")["revenue"].sum()` מחשב סכום הכנסה לכל מוצר. `.sort_values(ascending=False)` ממיין מהגבוה לנמוך. `.head(n)` לוקח את ה-n הראשונים. (אפשר גם `.nlargest(n)`, שעושה מיון ולקיחה ביחד.)

</div>

In [ ]:
def top_n_products(df, n=5):
    """df must already have a 'revenue' column.
    Returns a Series: index = product_name, value = total revenue of that product,
    sorted from highest to lowest, only the first n products.
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - שלב 2
def check_a2():
    t = pd.DataFrame({"product_name": ["a", "b", "a", "c"], "revenue": [5.0, 7.0, 4.0, 1.0]})
    r = top_n_products(t, 2)
    assert list(r.index) == ["a", "b"], "top_n_products: הדוגמה הקטנה - a (9) לפני b (7), וצריך להישאר 2 מוצרים"
    assert list(r.values) == [9.0, 7.0], "top_n_products: סכום ההכנסה של a הוא 5+4=9"
    big = top_n_products(add_revenue(orders), 5)
    assert list(big.index) == ["Plant Pot Plus", "Desk Organizer Basic", "Stapler Plus",
                               "Office Chair Mat Plus", "Storage Basket Pro"], \
        "top_n_products: על orders.csv המוצר הראשון צריך להיות Plant Pot Plus"
    return True

check_a2() and print("✓ שלב 2:")
print(top_n_products(add_revenue(orders), 5).round(0))

<div dir="rtl">

**זו התשובה הנכונה** של משימת העוגן, ובזכותה נוכל לבדוק כלים אחרים: 5 מוצרים, ממוינים מהגבוה לנמוך, הראשון **Plant Pot Plus** עם הכנסה של כ-33,596.

## שלב 3 - מבחן לתוצאה של כלי

עכשיו נכתוב את **המבחן** עצמו. הפונקציה `validate_top5` מקבלת תוצאה (`Series`) ש**כלי כלשהו** החזיר, ומחזירה **רשימה של בעיות** (טקסטים). רשימה ריקה = לא נמצאה בעיה.

ארבע בדיקות, אחת אחת (הוסיפו לרשימה `problems` בעזרת `problems.append("...")` רק כשהבדיקה נכשלת):

1. יש **בדיוק 5** מוצרים? (`len(result)`)
2. הרשימה **ממוינת מהגבוה לנמוך**? (השוו `list(result.values)` ל-`sorted(list(result.values), reverse=True)`)
3. המוצר הראשון הוא `Plant Pot Plus`? (`result.index[0]`)
4. ההכנסה שלו קרובה ל-33,596 (הפרש קטן מ-1)? (`abs(result.iloc[0] - 33596) < 1`)

</div>

In [ ]:
def validate_top5(result):
    """result: the Series that an AI-written function returned.
    Returns a list of problem descriptions (strings). An empty list means: no problem found.
    Checks: (1) exactly 5 items, (2) sorted from high to low,
            (3) first product is "Plant Pot Plus", (4) its revenue is within 1 of 33596.
    """
    problems = []
    # TODO: add a text to `problems` for every check that fails
    raise NotImplementedError

In [ ]:
# בדיקה - שלב 3
good = pd.Series([33596.0, 33379.0, 20522.0, 18829.0, 15467.0],
                 index=["Plant Pot Plus", "Desk Organizer Basic", "Stapler Plus",
                        "Office Chair Mat Plus", "Storage Basket Pro"])

def check_a3():
    assert validate_top5(good) == [], "validate_top5: על התוצאה הנכונה צריך להחזיר רשימה ריקה"
    assert len(validate_top5(good.iloc[:4])) >= 1, "validate_top5: 4 מוצרים במקום 5 צריך להיחשב בעיה"
    assert len(validate_top5(good.iloc[::-1])) >= 1, "validate_top5: סדר הפוך (מהנמוך לגבוה) צריך להיחשב בעיה"
    wrong_first = good.copy()
    wrong_first.index = ["Other"] + list(good.index[1:])
    assert len(validate_top5(wrong_first)) >= 1, "validate_top5: מוצר ראשון שגוי צריך להיחשב בעיה"
    return True

check_a3() and print("✓ שלב 3: המבחן עובד")

<div dir="rtl">

## שלב 4 - שלושה כלים, שלוש תשובות

בהרצאה הרצנו את אותה משימה בשלושה כלים. כאן מופיעה **דוגמה** של מה כל כלי **עשוי** להחזיר (שלוש פונקציות שכתבנו, בהשראת טעויות אופייניות. זה לא קובע איזה כלי טוב יותר: כל כלי יכול לטעות).

הריצו את התא (הוא רק מגדיר את הפונקציות) וקראו את הקוד של כל אחת:

</div>

In [ ]:
def ai_chat(df):
    # צ'אט שקיבל רק את שמות העמודות, בלי לדעת מהי "הכנסה"
    return df.groupby("product_name")["quantity"].sum().sort_values(ascending=False).head(5)

def ai_ide(df):
    # עוזר בעורך
    df = df.copy()
    df["revenue"] = df["quantity"] * df["unit_price"]
    return df.groupby("product_name")["revenue"].sum().head(5)

def ai_agent(df):
    # סוכן
    df = df.copy()
    df["revenue"] = df["quantity"] * df["unit_price"]
    return df.groupby("product_name")["revenue"].sum().nlargest(5)

print("✓ שלוש הפונקציות הוגדרו")

<div dir="rtl">

**לפני שמריצים את המבחן:** איזו מהשלוש נכונה? ולכל אחת מהשאר: מה לדעתכם הטעות? (קראו את הקוד.)

**התחזית שלי:** _(כתבו כאן)_

</div>

In [ ]:
for name, func in [("ai_chat", ai_chat), ("ai_ide", ai_ide), ("ai_agent", ai_agent)]:
    problems = validate_top5(func(orders))
    print(f"{name}:", "✓ לא נמצאו בעיות" if not problems else f"✗ {len(problems)} בעיות")
    for p in problems:
        print("    -", p)

<div dir="rtl">

**(א)** מה הייתה הטעות בכל אחד מהכלים הלא-נכונים? ומה הקשר בין הטעות של `ai_chat` לבין הכלל ש"בצ'אט, אתם הייתם ההקשר"? (רמז: מה הצ'אט קיבל בפרומפט, ומה הוא היה צריך לדעת?)

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

**(ב)** הגרף של `ai_chat` היה נראה תקין לגמרי: שמות מוצרים, 5 עמודות. מה זה מלמד על הכלל "הרצתי ולא קיבלתי שגיאה"? ומה בדיוק עושה `validate_top5` שהריצה לא עושה?

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

## שלב 5 - עכשיו בכלי אמיתי (בלי לדעת מראש מה יצא)

עד עכשיו עבדנו עם תשובות מוכנות. הגיע הזמן לנסות כלי אמיתי:

1. פתחו את **עוזר ה-AI של Colab** (כפתור ✨ **Gemini** בסרגל, או בלחיצה על "Generate" בתא קוד) - או כל כלי צ'אט שאתם משתמשים בו.
2. בקשו ממנו, באנגלית, את הדבר הבא (העתיקו את הפרומפט כמו שהוא):

```
I have a pandas DataFrame named orders with columns:
order_id, order_date, customer_id, product_id, product_name, quantity, unit_price.
Write a Python function top5(df) that returns a pandas Series: the 5 products
(by product_name) with the highest total revenue (quantity * unit_price),
sorted from high to low.
```

3. **הדביקו** את הפונקציה שהכלי כתב בתא הבא (אם הכלי קרא לה `top5`, שנו את השם ל-`my_ai_top5`), והריצו.
4. הריצו את `validate_top5` על התוצאה, ומלאו את הטבלה בתא שאחריו: **כמה פעמים העתקתם והדבקתם, כמה דקות לקח, וכמה בעיות המבחן מצא.**

> **שימו לב:** שאלו את עצמכם - מה הכלי **ראה** ומה **אתם** נאלצתם להעביר לו? (בצ'אט: אתם הייתם ההקשר.)

</div>

In [ ]:
# הדביקו כאן את הפונקציה שהכלי כתב (בשם my_ai_top5)
def my_ai_top5(df):
    raise NotImplementedError("הדביקו כאן את הקוד שהכלי כתב")

In [ ]:
my_problems = validate_top5(my_ai_top5(orders))
print("✓ לא נמצאו בעיות" if not my_problems else "✗ נמצאו בעיות:")
for p in my_problems:
    print("    -", p)

In [ ]:
# מלאו את הטבלה (החליפו את הערכים):
tool_log = {
    "tool": "(שם הכלי)",
    "copies": None,          # כמה פעמים העתקתם והדבקתם (מספר)
    "minutes": None,         # כמה דקות לקח עד שהיה קוד שרץ (מספר)
    "problems_found": None,  # כמה בעיות validate_top5 מצא (מספר)
}

In [ ]:
# בדיקה - שלב 5
def check_a5():
    assert tool_log["tool"] != "(שם הכלי)", "tool_log: כתבו את שם הכלי שבו השתמשתם"
    for key in ("copies", "minutes", "problems_found"):
        assert isinstance(tool_log[key], (int, float)), f"tool_log: הערך של {key} צריך להיות מספר"
    assert tool_log["problems_found"] == len(my_problems), "tool_log: problems_found צריך להתאים למה ש-validate_top5 מצא למעלה"
    return True

check_a5() and print("✓ שלב 5: התיעוד מוכן")

<div dir="rtl">

**(ג)** מה **אתם** נאלצתם להעביר לכלי ידנית (שמות עמודות? דוגמאות? הודעת שגיאה?), ומה הכלי ראה לבד? האם הכלי טעה? אם כן - באיזה מהשלבים הייתם מגלים זאת **בלי** `validate_top5`?

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

---

# חלק ב' - סוכן: מה הוא קרא, יצר והריץ (11 דק')

## הרקע

סוכן שורת פקודה הוא תוכנה שקוראת קבצים, מריצה פקודות ועורכת בעצמה. כדי לשלוט בו **צריך לקרוא את מה שהוא עשה**. הסוכנים מציגים "תמלול": רשימת פעולות. כאן נכתוב כלי שמנתח תמלול ועונה על שלוש שאלות: **מה נקרא? מה נוצר ושונה? מה הורץ?** ובנוסף: **כמה אישורים נדרשים**, ו**אילו פעולות מסוכנות**.

הפורמט (פעולה בכל שורה):

- `> read קובץ` - הסוכן קרא קובץ
- `> create קובץ` - הסוכן יצר קובץ
- `> edit קובץ` - הסוכן ערך קובץ קיים
- `> run פקודה` - הסוכן הריץ פקודה
- `> install חבילה` - הסוכן התקין חבילה
- `! הודעה` - שגיאה שהתקבלה
- `> done` - הסוכן סיים

הריצו וקראו את התמלול הראשון (סוכן שמבצע את משימת העוגן):

</div>

In [ ]:
transcript = """\
> read orders.csv
> read products.csv
> create analysis.py
> run python analysis.py
! KeyError: 'revenue'
> edit analysis.py
> run python analysis.py
> create top5.png
> done"""
print(transcript)

<div dir="rtl">

## שלב 1 - סיכום תמלול

ממשו `summarize_transcript(text)` שמחזירה מילון עם חמישה מפתחות: `read`, `created`, `edited`, `ran` (כולם **רשימות** של טקסטים) ו-`errors` (**מספר** שורות השגיאה).

**איך:** עוברים על כל שורה: `for line in text.splitlines():`. שורה שמתחילה ב-`"> read "` (בדקו עם `line.startswith(...)`) - מוסיפים את שארית השורה לרשימה `read`. את שארית השורה אפשר לקחת עם `line[len("> read "):]`. וכך הלאה לכל פעולה. שורה שמתחילה ב-`"! "` מגדילה את `errors` ב-1.

</div>

In [ ]:
def summarize_transcript(text):
    """text: an agent transcript (see format above).
    Returns {"read": [...], "created": [...], "edited": [...], "ran": [...], "errors": int}
    e.g. summarize_transcript("> read a.csv\n! oops") ->
         {"read": ["a.csv"], "created": [], "edited": [], "ran": [], "errors": 1}
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - שלב 1
def check_b1():
    assert summarize_transcript("> read a.csv\n! oops") == \
        {"read": ["a.csv"], "created": [], "edited": [], "ran": [], "errors": 1}, "summarize_transcript: הדוגמה מה-docstring"
    s = summarize_transcript(transcript)
    assert s["read"] == ["orders.csv", "products.csv"], "summarize_transcript: הסוכן קרא שני קבצים"
    assert s["created"] == ["analysis.py", "top5.png"], "summarize_transcript: נוצרו analysis.py ו-top5.png"
    assert s["edited"] == ["analysis.py"], "summarize_transcript: נערך analysis.py"
    assert s["ran"] == ["python analysis.py", "python analysis.py"], "summarize_transcript: הפקודה הורצה פעמיים"
    assert s["errors"] == 1, "summarize_transcript: הייתה שגיאה אחת"
    return True

check_b1() and print("✓ שלב 1:", summarize_transcript(transcript))

<div dir="rtl">

**(א)** בתמלול יש רגע שהסוכן **גילה בעצמו** שגיאה ותיקן אותה. מי שמבקש מצ'אט את אותה משימה - האם הוא היה מקבל את התיקון הזה בלי עזרתכם? אילו שורות בתמלול מראות את "בדיקה עצמית"?

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

## שלב 2 - כמה אישורים נדרשים?

בסולם האוטונומיה שלמדנו יש ארבע רמות. ברמה הראשונה (״רק מציע״) הסוכן לא פועל בעצמו, ולכן אין בה מה לאשר. בטבלה מופיעות שלוש הרמות האחרות (מהשליטה שלכם אל השליטה של הסוכן):

| רמה | משמעות | כמה אישורים |
|---|---|---|
| `"ask_all"` | שואל לפני כל פעולה: כל קובץ וכל פקודה | **כל** שורת פעולה `>` חוץ מ-`> done` |
| `"auto_edit"` | עריכות אוטומטיות, שואל רק על הרצת פקודות | רק שורות `> run` ו-`> install` |
| `"full_auto"` | פועל לבד עד הסוף | 0 |

ממשו `count_approvals(text, level)` שמחזירה כמה אישורים תצטרכו לתת בתמלול נתון.

**רמז:** עוברים על שורות הפעולה (`line.startswith("> ")` ו-`line != "> done"`), ולפי `level` מחליטים אם לספור אותה.

</div>

In [ ]:
def count_approvals(text, level):
    """level is "ask_all", "auto_edit" or "full_auto" (see the table above).
    Returns how many approvals the user must give for this transcript.
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - שלב 2
def check_b2():
    assert count_approvals(transcript, "ask_all") == 7, "count_approvals: ask_all - 7 פעולות בתמלול (בלי done)"
    assert count_approvals(transcript, "auto_edit") == 2, "count_approvals: auto_edit - רק שתי הרצות"
    assert count_approvals(transcript, "full_auto") == 0, "count_approvals: full_auto - אין אישורים"
    return True

check_b2() and print("✓ שלב 2:", {lvl: count_approvals(transcript, lvl) for lvl in ("ask_all", "auto_edit", "full_auto")})

<div dir="rtl">

**(ב)** ביום-יום, הרבה אנשים מעלים את האוטונומיה כי "נמאס לאשר". לפי הטבלה שהתקבלה - מה **חוסכים** בכל רמה, ומה **מפסידים**? באיזו רמה הייתם מתחילים בפרויקט שאין לו גיבוי?

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

## שלב 3 - זיהוי פעולות מסוכנות

התמלול הבא הוא מריצה אחרת. הפעם **הסוכן עשה יותר ממה שביקשתם**:

</div>

In [ ]:
transcript2 = """\
> read orders.csv
> edit analysis.py
> edit settings.py
> install fast-excel-writer-pro
> run rm -rf old_results
> run python analysis.py
> done"""
print(transcript2)

<div dir="rtl">

ממשו `find_risky(text, allowed_files)` שמחזירה **רשימה של שורות מסוכנות** (השורות המלאות, בסדר המקורי). שורה מסוכנת היא אחת מאלה:

1. התקנת חבילה - שורה שמתחילה ב-`"> install "`
2. הרצת פקודת מחיקה - שורת `> run` שמכילה את הטקסט `"rm "`
3. יצירה או עריכה של קובץ **שלא ברשימת הקבצים המותרים** - שורה שמתחילה ב-`"> edit "` או ב-`"> create "`, והקובץ שלה לא ב-`allowed_files`

</div>

In [ ]:
def find_risky(text, allowed_files):
    """Returns the list of risky lines (full lines, original order).
    Risky: an install; a run line that contains "rm "; an edit/create of a file not in allowed_files.
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - שלב 3
def check_b3():
    allowed = {"orders.csv", "analysis.py", "top5.png"}
    assert find_risky(transcript, allowed) == [], "find_risky: בתמלול הראשון אין פעולות מסוכנות"
    assert find_risky(transcript2, allowed) == [
        "> edit settings.py", "> install fast-excel-writer-pro", "> run rm -rf old_results"
    ], "find_risky: בתמלול השני שלוש שורות מסוכנות, בסדר שבו הן מופיעות"
    return True

check_b3() and print("✓ שלב 3:", find_risky(transcript2, {"orders.csv", "analysis.py", "top5.png"}))

<div dir="rtl">

**(ג)** התקנת `fast-excel-writer-pro` מזכירה משהו ממפגש 2. איזה סוג הזיה זה, ולמה התקנה אוטומטית של חבילה מסוכנת יותר מסתם **כתיבה** של קוד שמשתמש בה? באיזה כלל מהרשימה שלמעלה אפשר לתפוס את זה בקוד, ומה עוד צריך לעשות בעצמכם?

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

---

# חלק ג' - בחירת כלי: חמישה קריטריונים (10 דק')

## הרקע

בהרצאה למדנו לבחור כלי לפי **גודל המשימה, הקשר, סיכון, פרטיות ועלות, שליטה** - ולא לפי הרעש. עץ ההחלטה מהשקף:

- המשימה נוגעת בנתונים **רגישים**? ← **אל תשתמשו בכלי חיצוני**.
- המשימה בקובץ אחד, בלי הרצה? ← **עוזר בעורך** (אם הקובץ פתוח בעורך) או **צ'אט**.
- המשימה נוגעת בכמה קבצים או דורשת הרצה? ← אם יש **גיבוי**: **סוכן באוטונומיה נמוכה**. אם אין: **קודם גיבוי**.

כאן נהפוך את העץ לפונקציה, כדי שאפשר יהיה לבדוק אותו על תרחישים.

## שלב 1 - הפונקציה `choose_tool`

כל משימה מתוארת במילון עם חמישה שדות:

| שדה | משמעות |
|---|---|
| `files` | כמה קבצים המשימה נוגעת בהם (מספר) |
| `needs_run` | האם צריך להריץ קוד כדי לבדוק (`True`/`False`) |
| `sensitive` | האם יש נתונים רגישים (`True`/`False`) |
| `has_backup` | האם יש גיבוי ב-Git (`True`/`False`) |
| `open_in_editor` | האם הקובץ פתוח בעורך (`True`/`False`) |

הפונקציה מחזירה אחד מחמשת הטקסטים: `"no_external_tool"`, `"ide"`, `"chat"`, `"agent_low"`, `"backup_first"`.

**רמז:** שלוש בדיקות `if` לפי הסדר של העץ. **הסדר חשוב:** נתונים רגישים נבדקים **ראשונים**.

</div>

In [ ]:
def choose_tool(task):
    """task: dict with keys files, needs_run, sensitive, has_backup, open_in_editor.
    Returns one of: "no_external_tool", "ide", "chat", "agent_low", "backup_first".
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - חלק ג', שלב 1
def task(files, needs_run, sensitive, has_backup, open_in_editor):
    return dict(files=files, needs_run=needs_run, sensitive=sensitive, has_backup=has_backup, open_in_editor=open_in_editor)

def check_c1():
    cases = [
        ("להבין שגיאה מוזרה", task(0, False, False, False, False), "chat"),
        ("להשלים פונקציה בקובץ פתוח", task(1, False, False, False, True), "ide"),
        ("פיצ'ר ב-5 קבצים ובדיקות, יש גיבוי", task(5, True, False, True, False), "agent_low"),
        ("אותו פיצ'ר, אין גיבוי", task(5, True, False, False, False), "backup_first"),
        ("לנתח נתוני לקוחות אמיתיים", task(3, True, True, True, False), "no_external_tool"),
        ("לשנות צבע של כפתור באתר, קובץ אחד פתוח", task(1, False, False, False, True), "ide"),
    ]
    for name, t, expected in cases:
        got = choose_tool(t)
        assert got == expected, f"choose_tool: ״{name}״ - ציפינו ל-{expected}, התקבל {got}"
    return True

check_c1() and print("✓ חלק ג' שלב 1: העץ עובד על 6 תרחישי ההרצאה")

<div dir="rtl">

## שלב 2 - תרחישים משלכם

כתבו **שלושה תרחישים** מהחיים שלכם (מהעבודה, מהלימודים או מהפרויקט) בצורת מילון - בעזרת הפונקציה `task(...)` שכבר הוגדרה. התא יראה לכם איזה כלי בחרה הפונקציה. **אם התוצאה לא מסתדרת לכם עם האינטואיציה** - זה בדיוק מה שצריך לדון בו.

</div>

In [ ]:
my_tasks = {
    # "תיאור קצר של תרחיש": task(files, needs_run, sensitive, has_backup, open_in_editor),
    # לדוגמה: "לתקן שגיאת כתיב בקובץ README": task(1, False, False, False, True),
}

for name, t in my_tasks.items():
    print(f"{name}  ←  {choose_tool(t)}")

In [ ]:
# בדיקה - חלק ג', שלב 2
def check_c2():
    assert len(my_tasks) >= 3, "my_tasks: כתבו לפחות שלושה תרחישים"
    for name, t in my_tasks.items():
        assert set(t) == {"files", "needs_run", "sensitive", "has_backup", "open_in_editor"}, f"my_tasks: ב-״{name}״ חסרים שדות"
    return True

check_c2() and print("✓ חלק ג' שלב 2:", len(my_tasks), "תרחישים")

<div dir="rtl">

## שלב 3 - מה מותר להעלות?

אחד מחמשת הקריטריונים הוא **פרטיות**: מה נשלח לשרתי הספק? גם סוכן שרץ "אצלכם" שולח לשרתי הספק את הפרומפט ואת הקבצים שהוא קרא. הרמזור מההרצאה:

- `"red"` (🔴 לא להעלות): נתוני לקוחות, מפתחות וסיסמאות, קוד סודי של ארגון
- `"yellow"` (🟡 בזהירות): קוד של הפרויקט שלכם, אחרי בדיקת תנאי השימוש
- `"green"` (🟢 בסדר): נתונים סינתטיים, דוגמאות, קוד ציבורי

סווגו כל פריט: החליפו את `"?"` בצבע המתאים.

</div>

In [ ]:
data_level = {
    "orders.csv הסינתטי של הקורס": "?",
    "קוד הפרויקט הקבוצתי שלכם": "?",
    "קוד פתוח שהורדתם מ-GitHub": "?",
    "סיסמת מסד הנתונים של הארגון": "?",
    "קובץ לקוחות אמיתי מהעבודה": "?",
    "מפתח API של שירות מסחרי": "?",
}

In [ ]:
# בדיקה - חלק ג', שלב 3
def check_c3():
    expected = {
        "orders.csv הסינתטי של הקורס": "green",
        "קוד הפרויקט הקבוצתי שלכם": "yellow",
        "קוד פתוח שהורדתם מ-GitHub": "green",
        "סיסמת מסד הנתונים של הארגון": "red",
        "קובץ לקוחות אמיתי מהעבודה": "red",
        "מפתח API של שירות מסחרי": "red",
    }
    for item, level in expected.items():
        assert data_level.get(item) in ("red", "yellow", "green"), f"data_level: ״{item}״ - כתבו red / yellow / green"
        assert data_level[item] == level, f"data_level: ״{item}״ - חשבו שוב: מה קורה אם זה יישלח לשרתי הספק?"
    return True

check_c3() and print("✓ חלק ג' שלב 3: סיווג הפרטיות נכון")

<div dir="rtl">

## כלל ההחלטה שלכם

</div>

<div dir="rtl">

השלימו את המשפט בעזרת מה שבנינו: **כשהמשימה ___ וההקשר ___ והסיכון ___, אבחר ב-___ כי ___.** (משפט אחד שהוא כלל **שלכם**, שאפשר לקחת לפרויקט הקבוצתי.)

**הכלל שלי:** _(כתבו כאן)_

</div>

<div dir="rtl">

---

# חלק ד' - יומן שימוש ב-AI (3 דק')

## הרקע

בפרויקט הקבוצתי כל שימוש משמעותי ב-AI נרשם ביומן: **כלי ופרומפט ← מה קיבלתי ← איך אימתתי ← החלטה (אימצתי / דחיתי / תיקנתי)**. ביומן אפשר להוכיח **אחריות אישית**, והוא עוזר לכם לזכור מה בדקתם.

היומן הוא קובץ `AI_LOG.md` בפורמט טבלה. נכתוב שתי פונקציות:

- `log_row(tool, prompt, got, verified, decision)` ← מחזירה **שורה אחת** של טבלה, בפורמט `| כלי | פרומפט | ... |`. אם בטקסט מופיע התו `|`, מחליפים אותו ב-`/` כדי לא לשבור את הטבלה.
- `write_log(rows, path)` ← כותבת קובץ: שורת כותרת, שורת הפרדה `|---|---|---|---|---|`, ואז את כל השורות. מחזירה את `path`.

**כלי:** `" | ".join(רשימה)` מחבר טקסטים עם `|` ביניהם. `text.replace("|", "/")` מחליף תו. `open(path, "w", encoding="utf-8")` ו-`f.write(...)` כותבים קובץ.

</div>

In [ ]:
def log_row(tool, prompt, got, verified, decision):
    """Returns one markdown table row: "| tool | prompt | got | verified | decision |"
    Any "|" inside a text is replaced by "/" so the table does not break.
    """
    # TODO
    raise NotImplementedError


def write_log(rows, path="AI_LOG.md"):
    """Writes the header line, the separator line "|---|---|---|---|---|" and then all rows (one per line).
    Returns path.
    """
    header = "| כלי | פרומפט | מה קיבלתי | איך אימתתי | החלטה |"
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - חלק ד'
def check_d():
    r = log_row("chat", "top 5 | by revenue", "pandas code", "ran it", "adopted")
    assert r == "| chat | top 5 / by revenue | pandas code | ran it | adopted |", "log_row: הפורמט, והחלפת | ב-/"
    p = write_log([r, r], "AI_LOG_test.md")
    lines = open(p, encoding="utf-8").read().splitlines()
    assert len(lines) == 4, "write_log: צריכות להיות 4 שורות - כותרת, הפרדה ושתי שורות נתונים"
    assert lines[1] == "|---|---|---|---|---|", "write_log: שורה שנייה היא |---|---|---|---|---|"
    assert lines[0].count("|") == 6, "write_log: בשורת הכותרת צריכות להיות חמש עמודות (6 תווי |)"
    return True

check_d() and print("✓ חלק ד': הפונקציות עובדות")

<div dir="rtl">

עכשיו נרשום ביומן את הניסוי של חלק א' (שלב 5) ואת התשובה של `ai_chat`, ונראה את הקובץ:

</div>

In [ ]:
rows = [
    log_row(tool_log["tool"], "top5(df): 5 products by revenue", "פונקציה שהודבקה ל-Colab",
            f"validate_top5 מצא {tool_log['problems_found']} בעיות; העתקות: {tool_log['copies']}", "(כתבו: אימצתי / דחיתי / תיקנתי)"),
    log_row("(כלי)", "top 5 products (בלי להגדיר הכנסה)", "ai_chat: סכום כמויות", "validate_top5: מוצר ראשון שגוי", "דחיתי: הוספתי את הנוסחה לפרומפט"),
]
print(open(write_log(rows), encoding="utf-8").read())

<div dir="rtl">

הקובץ `AI_LOG.md` נשמר בחלונית הקבצים של Colab (📁 בצד שמאל). **להמשך הפרויקט:** הורידו אותו (קליק ימני ← Download) והעלו למאגר הקבוצתי ב-GitHub, בהתאם לרשימת הקבוצות על המקרן. (פתיחת קבוצה ומאגר, ותיעוד ב-`AI_LOG.md`, הן משימת ההכנה למפגש 6.)

</div>

<div dir="rtl">

---

# חלק ה' - מסקנה (3 דק')

ארבעה משפטים, אחד לכל חלק. כל משפט בנוי כך: **מה מימשתם ומדדתם ← מה זה מראה על הכלים ← מה עושים בעקבות זה בעבודה עם AI.**

1. **בדיקת תוצאה:** _(כתבו כאן)_
2. **סוכן:** _(כתבו כאן)_
3. **בחירת כלי:** _(כתבו כאן)_
4. **יומן ופרטיות:** _(כתבו כאן)_

</div>

<div dir="rtl">

---

# בונוס

**1. עברית מול אנגלית.** בקשו מהכלי מחלק א' את אותה משימה **בעברית** (תרגמו את הפרומפט). האם הקוד זהה? האם `validate_top5` מצא בעיות? מה זה אומר על מה שלמדנו במפגש 2 על שפה וטוקנים?

**2. הרצה חוזרת.** הריצו את אותו פרומפט פעמיים בשיחות חדשות. האם קיבלתם אותו קוד? (זכרו את הטריק ממפגש 2: תשובות שונות הן סימן לאי-ודאות.)

**3. גרף.** השתמשו ב-`plt.barh` כדי להציג את חמשת המוצרים: `top_n_products(add_revenue(orders)).plot.barh()`. האם הוא נראה כמו הגרף של `ai_chat`? למה הגרף בלבד לא הספיק כדי לגלות את הטעות?

</div>

In [ ]:
# תא לבונוס 1

In [ ]:
# תא לבונוס 2

In [ ]:
# תא לבונוס 3

<div dir="rtl">

---

# העשרה (אופציונלי, ~10 דק', בית): כלי SQL בסגנון MCP, מוגבל ובטוח יותר

## הרקע

במצגת ראינו ש-**MCP** מחבר סוכן למקור נתונים (למשל מסד נתונים), ושהסוכן יכול לנסח בעצמו שאילתות SQL בשבילכם. זה חזק, וגם מסוכן: **שאילתה אחת שגויה (או שאילתה שתוקף הכניס לטקסט) יכולה למחוק טבלה או לדלוף עמודות רגישות.** הכלל מהמפגש: **הרשאה מינימלית**. כאן נבנה כלי קטן שמתנהג כמו שרת MCP למסד נתונים, עם **שלוש שכבות הגנה**:

1. **סינון בקוד:** רק שאילתת `SELECT` אחת, ורק על טבלאות מותרות.
2. **הרשאות במסד עצמו:** חיבור לקריאה בלבד, שמונע גישה לטבלה הרגישה.
3. **יומן ביקורת:** כל שאילתה נרשמת (הותרה או נחסמה).

**כנות:** סינון טקסט הוא שכבה **חלשה** (נראה בעוד רגע איך עוקפים אותו). ההגנה האמיתית היא ההרשאות במסד. לכן בונים את שתיהן.

הנתונים: טבלת `orders` (3,000 ההזמנות), וטבלה סינתטית `customers` עם `phone` ו-`email` (**מידע רגיש**), ותצוגה `customers_safe` שחושפת רק `customer_id` ו-`city`.

</div>

In [ ]:
import os, re, sqlite3

DB = os.path.abspath("store_enrichment.db")
if os.path.exists(DB):
    os.remove(DB)

def _build_db():
    con = sqlite3.connect(DB)
    orders.drop(columns=["revenue"], errors="ignore").to_sql("orders", con, index=False)
    ids = sorted(orders["customer_id"].unique())
    cities = ["Haifa", "Tel Aviv", "Ariel", "Beer Sheva"]
    rows = [(c, f"050-{int(c[1:]):07d}", f"{c.lower()}@example.test", cities[int(c[1:]) % 4]) for c in ids]
    con.execute("CREATE TABLE customers (customer_id TEXT PRIMARY KEY, phone TEXT, email TEXT, city TEXT)")
    con.executemany("INSERT INTO customers VALUES (?, ?, ?, ?)", rows)
    con.execute("CREATE VIEW customers_safe AS SELECT customer_id, city FROM customers")   # בלי phone ו-email
    con.commit()
    con.close()

def _authorizer(action, arg1, arg2, dbname, source):
    """ההרשאה ברמת המסד: אסור לקרוא מהטבלה customers, אלא דרך התצוגה customers_safe."""
    if action == sqlite3.SQLITE_READ and arg1 == "customers" and source != "customers_safe":
        return sqlite3.SQLITE_DENY
    return sqlite3.SQLITE_OK

def read_only_connection(with_authorizer=True):
    """חיבור לקריאה בלבד (mode=ro). עם with_authorizer=True גם נחסמת הקריאה מ-customers."""
    con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)
    if with_authorizer:
        con.set_authorizer(_authorizer)
    return con

_build_db()
ALLOWED_TABLES = {"orders", "customers_safe"}     # הטבלאות שהכלי מותר לו לקרוא
AUDIT = []                                         # יומן ביקורת: (שאילתה, "allowed"/"blocked", סיבה)
print("✓ המסד נבנה. טבלאות מותרות לכלי:", sorted(ALLOWED_TABLES))

<div dir="rtl">

## שלב 1 - רק שאילתת SELECT אחת

ממשו `is_read_only(sql)`: מחזירה `True` רק אם הטקסט הוא **שאילתת SELECT אחת**, ובלי מילים שמשנות נתונים. התנאים (כולם חייבים להתקיים):

1. אחרי ניקוי רווחים וסימן `;` בסוף, הטקסט **מתחיל ב-`select`** (בלי תלות באותיות גדולות/קטנות).
2. אין בו `;` נוסף (שאילתה שנייה מוסתרת, כמו `SELECT 1; DELETE ...`).
3. אין בו אף אחת מהמילים `FORBIDDEN` (מילה שלמה, בלי תלות באותיות).

**כלים:** `sql.strip().rstrip(";").strip().lower()`; `text.startswith("select")`; `";" in text`; `set(re.findall(r"[a-z_]+", text))` נותן את קבוצת המילים; `words & FORBIDDEN` הוא החיתוך (קבוצה ריקה = אין מילה אסורה).

</div>

In [ ]:
FORBIDDEN = {"insert", "update", "delete", "drop", "alter", "create", "attach", "pragma", "replace"}

def is_read_only(sql):
    """True only for ONE select statement without any word from FORBIDDEN.
    e.g. is_read_only("SELECT * FROM orders;") -> True ;  is_read_only("DROP TABLE orders") -> False
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - העשרה, שלב 1
def check_x1():
    assert is_read_only("SELECT * FROM orders") is True, "is_read_only: SELECT רגיל צריך לעבור"
    assert is_read_only("  select product_name from orders;  ") is True, "is_read_only: רווחים, אותיות קטנות ו-; בסוף - עדיין SELECT"
    assert is_read_only("SELECT created_at FROM orders") is True, "is_read_only: created_at היא מילה אחת, לא המילה create"
    assert is_read_only("DROP TABLE orders") is False, "is_read_only: DROP חייב להיחסם"
    assert is_read_only("SELECT 1; DELETE FROM orders") is False, "is_read_only: שאילתה שנייה מוסתרת אחרי ; חייבת להיחסם"
    assert is_read_only("UPDATE orders SET quantity = 0") is False, "is_read_only: UPDATE חייב להיחסם"
    assert is_read_only("select * from orders where 1=1 -- ; drop") is False, "is_read_only: גם drop בתוך הערה נחסם (עדיף להיזהר)"
    return True

check_x1() and print("✓ העשרה שלב 1: הסינון הראשון עובד")

<div dir="rtl">

## שלב 2 - רק טבלאות מותרות

ממשו `tables_not_allowed(sql, allowed)`: מחזירה **רשימה ממוינת** של שמות הטבלאות שהשאילתה קוראת מהן ואינן ב-`allowed`. טבלה "נקראת" כשהיא מופיעה מיד אחרי `FROM` או `JOIN`.

**כלים:** `re.findall(r"\b(?:from|join)\s+(\w+)", sql.lower())` מחזירה את שמות הטבלאות. `set(...)` הופך לקבוצה, ו-`used - set(allowed)` נותן מה שלא מותר. `sorted(...)` ממיין.

</div>

In [ ]:
def tables_not_allowed(sql, allowed):
    """Returns a sorted list of table names that appear right after FROM or JOIN and are not in `allowed`.
    e.g. tables_not_allowed("SELECT * FROM customers", {"orders"}) -> ["customers"]
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - העשרה, שלב 2
def check_x2():
    ok = {"orders", "customers_safe"}
    assert tables_not_allowed("SELECT * FROM orders", ok) == [], "tables_not_allowed: orders מותרת - רשימה ריקה"
    assert tables_not_allowed("SELECT * FROM customers", ok) == ["customers"], "tables_not_allowed: customers אינה מותרת"
    assert tables_not_allowed("select o.order_id from orders o JOIN customers c ON c.customer_id = o.customer_id", ok) == ["customers"], \
        "tables_not_allowed: גם טבלה אחרי JOIN נבדקת"
    assert tables_not_allowed("SELECT * FROM Customers", ok) == ["customers"], "tables_not_allowed: בלי תלות באותיות גדולות/קטנות"
    assert tables_not_allowed("SELECT * FROM customers_safe JOIN orders USING (customer_id)", ok) == [], \
        "tables_not_allowed: customers_safe היא שם מלא, שונה מ-customers"
    return True

check_x2() and print("✓ העשרה שלב 2: בדיקת הטבלאות עובדת")

<div dir="rtl">

## שלב 3 - הכלי המלא: `safe_query`

חברו את הכול לפונקציה אחת, כמו כלי בשרת MCP. `safe_query(sql, max_rows=20)`:

1. אם `is_read_only(sql)` נכשלת: רשמו ב-`AUDIT` שורה `(sql, "blocked", "not read-only")` וזרקו `PermissionError("הכלי מריץ רק שאילתת SELECT אחת")`.
2. אם `tables_not_allowed(sql, ALLOWED_TABLES)` מחזירה רשימה לא ריקה: רשמו `(sql, "blocked", "table not allowed")` וזרקו `PermissionError` עם שמות הטבלאות.
3. אחרת: רשמו `(sql, "allowed", "")`, והריצו את השאילתה **עם הגבלת שורות** והחזירו טבלה. ההגבלה: עוטפים `f"SELECT * FROM ({sql_ללא_;}) LIMIT {max_rows}"`.

**כלים:** `AUDIT.append((sql, "blocked", "..."))`; `raise PermissionError("...")`; `pd.read_sql_query(שאילתה, read_only_connection())` מריצה ומחזירה טבלה.

</div>

In [ ]:
def safe_query(sql, max_rows=20):
    """A guarded SQL tool: one SELECT only, only allowed tables, at most max_rows rows, everything logged in AUDIT.
    Raises PermissionError when blocked. Returns a pandas table otherwise.
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - העשרה, שלב 3
def check_x3():
    start = len(AUDIT)
    good = safe_query("SELECT product_name, SUM(quantity * unit_price) AS revenue FROM orders GROUP BY product_name ORDER BY revenue DESC")
    assert len(good) == 20, "safe_query: ברירת המחדל max_rows=20 - צריכות לחזור 20 שורות"
    assert good.iloc[0, 0] == "Plant Pot Plus", "safe_query: המוצר הראשון צריך להיות Plant Pot Plus (כמו במחברת)"
    assert len(safe_query("SELECT * FROM orders", max_rows=3)) == 3, "safe_query: max_rows=3 צריך להחזיר 3 שורות בלבד"
    safe_query("SELECT * FROM customers_safe", max_rows=2)
    attacks = ["DROP TABLE orders", "SELECT 1; DELETE FROM orders", "SELECT * FROM customers", "SELECT phone FROM customers"]
    for q in attacks:
        try:
            safe_query(q)
        except PermissionError:
            pass
        else:
            raise AssertionError(f"safe_query: השאילתה הזו הייתה צריכה להיחסם: {q}")
    statuses = [row[1] for row in AUDIT[start:]]
    assert statuses == ["allowed", "allowed", "allowed"] + ["blocked"] * 4, f"safe_query: יומן הביקורת צריך לרשום 3 הותרו ו-4 נחסמו, ונרשם: {statuses}"
    assert len(pd.read_sql_query("SELECT * FROM orders", sqlite3.connect(DB))) == 3000, "safe_query: הטבלה orders נפגעה"
    return True

check_x3() and print("✓ העשרה שלב 3: הכלי המוגן עובד")

<div dir="rtl">

## עכשיו מנסים לשבור אותו

**לפני שמריצים:** מבין שש השאילתות הבאות (חלק מהן הסוכן עלול לכתוב בטעות, וחלק מוכנס ע"י תוקף בתוך טקסט של מייל או מסמך), אילו ייחסמו? ובאיזו שכבה? שימו לב במיוחד לשאילתה האחרונה (`FROM orders, customers` - עם פסיק).

</div>

<div dir="rtl">

**התחזית שלי:** _(כתבו כאן)_

</div>

In [ ]:
TESTS = [
    "SELECT product_name, SUM(quantity * unit_price) AS revenue FROM orders GROUP BY 1 ORDER BY 2 DESC",
    "DROP TABLE orders",
    "SELECT 1; DELETE FROM orders",
    "SELECT * FROM customers",
    "SELECT o.order_id, c.phone FROM orders o JOIN customers c ON c.customer_id = o.customer_id",
    "SELECT order_id, phone FROM orders, customers WHERE orders.customer_id = customers.customer_id",
]
for q in TESTS:
    try:
        df = safe_query(q, max_rows=3)
        print("✓ הותרה:", q[:70], "->", len(df), "שורות")
    except PermissionError as e:
        print("✗ נחסמה בסינון:", q[:70], "|", e)
    except NotImplementedError:
        print("עדיין לא מימשתם את safe_query (שלב 3). השלימו אותה והריצו שוב.")
        break
    except Exception as e:
        print("🛡 הסינון הותיר, אבל המסד חסם:", q[:60], "|", type(e).__name__, e)

<div dir="rtl">

**ראיתם את זה:** שאילתה עם **פסיק** (`FROM orders, customers`) עוברת את הסינון הטקסטואלי, אבל **המסד חוסם אותה** בגלל ההרשאה (`_authorizer`) - זו שכבה 2. נראה מה היה קורה בלי שכבה 2: אותו חיבור לקריאה בלבד, אבל **בלי** ההרשאות על הטבלה:

</div>

In [ ]:
leaky = pd.read_sql_query(
    "SELECT order_id, phone FROM orders, customers WHERE orders.customer_id = customers.customer_id LIMIT 3",
    read_only_connection(with_authorizer=False))
print("בלי הרשאות ברמת המסד, הסינון בקוד לא הספיק. הדלפנו (נתוני דמה):")
print(leaky)

try:
    read_only_connection().execute("DELETE FROM orders")
except Exception as e:
    print("\nגם אם הסינון נעקף, חיבור לקריאה בלבד אינו מאפשר למחוק:", type(e).__name__, "-", e)

<div dir="rtl">

## ההודעות עצמן: איך סוכן ושרת MCP מדברים

עד עכשיו ראינו את **הכלי** (`safe_query`). השאלה: איך הסוכן **מגיע** אליו? ב-MCP הסוכן (ה-**host**, ובתוכו ה-**client**) והשרת מחליפים **הודעות JSON** (בפרוטוקול JSON-RPC 2.0). בשרת אמיתי ההודעות עוברות כשורות טקסט דרך `stdin/stdout` (תהליך מקומי) או דרך HTTP (שרת מרוחק). כאן נדמה את זה בפונקציה אחת, `mcp_handle`, כדי שתראו **את ההודעות בעצמן**. (זו הדמיה פשוטה לצורכי לימוד. הפרוטוקול האמיתי מכיל עוד שדות; הפרטים המלאים במפרט הרשמי.)

**חשוב לדעת:** ההדמיה עוקבת אחרי הדגם של גרסאות המפרט עד `2025-11-25`, הכולל לחיצת יד (`initialize`). בגרסה העדכנית (`2026-07-28`) לחיצת היד הוסרה: כל בקשה נושאת את גרסת הפרוטוקול ואת יכולות הלקוח בשדה `_meta`, ויש שיטה `server/discover` שבה השרת מציג את עצמו. הרעיון נשאר זהה: גילוי כלים ואז הפעלתם. המפרט משתנה, ולכן בפרויקט בודקים תמיד את הגרסה העדכנית ב־modelcontextprotocol.io.

הסדר בכל חיבור:

1. `initialize`: לחיצת יד. הלקוח אומר איזו גרסה הוא מכיר, והשרת עונה מה הוא יודע לעשות (`capabilities`).
2. `tools/list`: **גילוי כלים.** השרת מחזיר רשימה: לכל כלי `name`, `description` ו-`inputSchema` (מה הפרמטרים). **זה כל מה שהמודל רואה על הכלי.**
3. `tools/call`: **הפעלת כלי.** המודל מחליט לקרוא לכלי (`name` + `arguments`), והשרת מריץ (כאן: `safe_query` על מסד הנתונים) ומחזיר `content`. אם הכלי נחסם, חוזר `isError: true` עם הסבר, **והמודל קורא אותו וממשיך**.

</div>

In [ ]:
import json

TOOLS = [{
    "name": "query",
    "description": "Run ONE read-only SELECT on the store database. Tables: orders (order_id, order_date, customer_id, product_id, product_name, quantity, unit_price) and customers_safe (customer_id, city). Revenue = quantity * unit_price. Max 5 rows.",
    "inputSchema": {"type": "object",
                    "properties": {"sql": {"type": "string", "description": "a single SELECT statement"}},
                    "required": ["sql"]},
}]

def mcp_handle(req):
    """A tiny MCP-style server: gets a JSON-RPC request (dict), returns the response (dict) or None for notifications."""
    if "id" not in req:                       # הודעה בלי id = notification: אין תשובה
        return None
    method, rid = req["method"], req["id"]
    if method == "initialize":
        result = {"protocolVersion": "2025-06-18", "capabilities": {"tools": {}},
                  "serverInfo": {"name": "store-db", "version": "0.1"}}
    elif method == "tools/list":
        result = {"tools": TOOLS}
    elif method == "tools/call":
        params = req["params"]
        if params["name"] != "query":
            return {"jsonrpc": "2.0", "id": rid, "error": {"code": -32602, "message": f"Unknown tool: {params['name']}"}}
        try:
            text, is_error = safe_query(params["arguments"]["sql"], max_rows=5).to_csv(index=False), False
        except PermissionError as e:
            text, is_error = f"blocked: {e}", True
        except NotImplementedError:
            text, is_error = "tool not implemented yet (finish safe_query, step 3)", True
        result = {"content": [{"type": "text", "text": text}], "isError": is_error}
    else:
        return {"jsonrpc": "2.0", "id": rid, "error": {"code": -32601, "message": f"Method not found: {method}"}}
    return {"jsonrpc": "2.0", "id": rid, "result": result}

CONVERSATION = [
    {"jsonrpc": "2.0", "id": 1, "method": "initialize",
     "params": {"protocolVersion": "2025-06-18", "capabilities": {}, "clientInfo": {"name": "my-agent", "version": "1.0"}}},
    {"jsonrpc": "2.0", "method": "notifications/initialized"},
    {"jsonrpc": "2.0", "id": 2, "method": "tools/list"},
    {"jsonrpc": "2.0", "id": 3, "method": "tools/call",
     "params": {"name": "query", "arguments": {"sql": "SELECT product_name, SUM(quantity * unit_price) AS revenue FROM orders GROUP BY 1 ORDER BY 2 DESC"}}},
    {"jsonrpc": "2.0", "id": 4, "method": "tools/call",
     "params": {"name": "query", "arguments": {"sql": "SELECT * FROM customers"}}},
]
for msg in CONVERSATION:
    print("→ הסוכן (client) שולח:\n" + json.dumps(msg, ensure_ascii=False, indent=2))
    reply = mcp_handle(msg)
    print("← השרת (MCP server) עונה:\n" + (json.dumps(reply, ensure_ascii=False, indent=2) if reply else "(notification: אין תשובה)"))
    print("-" * 60)

<div dir="rtl">

שימו לב מה קרה בהודעה האחרונה (`SELECT * FROM customers`): השרת **לא קרס**. הוא החזיר `isError: true` עם הסבר, והסוכן יכול לקרוא אותו ולנסות שאילתה אחרת. ושימו לב ל-`description` של הכלי: **הוא כתוב כך שהמודל ידע** אילו טבלאות קיימות, מה הן העמודות, ואיך מחשבים הכנסה. כך "מלמדים" סוכן מראש מה לחפש.

</div>

<div dir="rtl">

**(א)** למה הסינון בקוד (`is_read_only`, `tables_not_allowed`) **לא מספיק**, ומה בכל זאת הוא נותן? מה הוסיפה שכבה 2 (הרשאות במסד)? נסחו בעזרת המושג **הרשאה מינימלית** מהמצגת.

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

**(ב)** מה הכלי הזה **לא** מגן עליו, גם עם שתי השכבות? (רמזים: מה אם `orders` עצמה מכילה מידע רגיש? מה אם הסוכן מקבל טקסט זדוני מתוך הנתונים? מה אם הסוכן שולח שאילתות כבדות שוב ושוב?) ומה עוד הייתם מוסיפים בארגון אמיתי?

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

**(ג)** הסוכן מעולם לא ראה את הקוד של `safe_query`. מאיפה הוא יודע שיש כלי בשם `query`, מה לשלוח אליו, ובאיזה פורמט להחזיר? ומה קורה לדיוק של השאילתות אם ה-`description` מעורפל (למשל "שאילתה על המסד")? חברו למה שלמדנו על פרומפטים והקשר (מפגשים 3 ו-4, לפי הסילבוס).

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

---

# בדיקה מסכמת - הריצו לפני ההגשה

התא הבא מריץ שוב את כל תאי הבדיקה. לפני ההגשה: **Runtime ← Restart session and run all**, וודאו שכל השורות כאן מסומנות `✓`.

</div>

In [ ]:
for name, check in [("א' - add_revenue", check_a1),
                    ("א' - top_n_products", check_a2),
                    ("א' - validate_top5", check_a3),
                    ("א' - תיעוד הכלי האמיתי", check_a5),
                    ("ב' - summarize_transcript", check_b1),
                    ("ב' - count_approvals", check_b2),
                    ("ב' - find_risky", check_b3),
                    ("ג' - choose_tool", check_c1),
                    ("ג' - התרחישים שלי", check_c2),
                    ("ג' - סיווג פרטיות", check_c3),
                    ("ד' - log_row, write_log", check_d),
                    ("העשרה (אופציונלי) - is_read_only", check_x1),
                    ("העשרה (אופציונלי) - tables_not_allowed", check_x2),
                    ("העשרה (אופציונלי) - safe_query", check_x3)]:
    try:
        check()
        print(f"✓ {name}")
    except NotImplementedError:
        print(f"✗ {name}  ← עדיין לא מומש")
    except AssertionError as e:
        print(f"✗ {name}  ← {e}")